<a href="https://colab.research.google.com/github/Festus004/sece-ai-rag-pipeline/blob/main/SECE_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from bs4 import BeautifulSoup

print("Initializing SECE Web Scraper...")

# A list of high-value target URLs from the Sri Eshwar College of Engineering website
urls = [
    "https://sece.ac.in/",                                     # Home / General Info
    "https://sece.ac.in/pages/about-us",                       # Vision, Mission & History
    "https://sece.ac.in/pages/computer-science-and-engineering",# CSE Department details
    "https://sece.ac.in/pages/placements"                      # Placement statistics & recruiters
]

# Set a browser header so the website allows our automated python request safely
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

scraped_text_corpus = ""

for url in urls:
    try:
        print(f"Scraping context from: {url}")
        response = requests.get(url, headers=headers, timeout=15)

        if response.status_code == 200 or response.ok:
            soup = BeautifulSoup(response.text, 'html.parser')

            # Clean up the page source by dropping unnecessary script and style tags
            for script in soup(["script", "style", "nav", "footer"]):
                script.decompose()

            # Extract clean, human-readable text blocks
            page_text = soup.get_text(separator="\n")

            # Remove excessive empty white spacing lines
            clean_lines = [line.strip() for line in page_text.splitlines() if line.strip()]
            scraped_text_corpus += f"\n\n--- Source URL: {url} ---\n" + "\n".join(clean_lines)

    except Exception as e:
        print(f"Skipping {url} due to connection handling: {e}")

# Save the gathered raw text knowledge base into a local file
output_filename = "sece_data.txt"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(scraped_text_corpus)

print(f"\nStep 1 Complete! Clean text extracted and saved successfully to '{output_filename}'.")

Initializing SECE Web Scraper...
Scraping context from: https://sece.ac.in/
Scraping context from: https://sece.ac.in/pages/about-us
Scraping context from: https://sece.ac.in/pages/computer-science-and-engineering
Scraping context from: https://sece.ac.in/pages/placements

Step 1 Complete! Clean text extracted and saved successfully to 'sece_data.txt'.


In [3]:
!pip install -q langchain-text-splitters

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Loading raw SECE knowledge base text...")
# 1. Read the text file we scraped in Step 1
with open("sece_data.txt", "r", encoding="utf-8") as f:
    raw_college_data = f.read()

# 2. Configure the recursive splitter with specific chunk targets
# chunk_size: 1000 characters per text block
# chunk_overlap: 200 characters to ensure sentence continuity
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

print("Executing split algorithm across the corpus...")
# 3. Perform the mathematical split operation
sece_chunks = text_splitter.create_documents([raw_college_data])

print(f"\nStep 2 Complete!")
print(f"Total raw text length: {len(raw_college_data)} characters.")
print(f"Successfully generated {len(sece_chunks)} individual text chunks!")

# Let's print out the very first chunk so you can see what it looks like
print("\n--- SAMPLE CHUNK 1 LOOKAHEAD ---")
print(sece_chunks[0].page_content)
print("---------------------------------")

Loading raw SECE knowledge base text...
Executing split algorithm across the corpus...

Step 2 Complete!
Total raw text length: 24993 characters.
Successfully generated 32 individual text chunks!

--- SAMPLE CHUNK 1 LOOKAHEAD ---
--- Source URL: https://sece.ac.in/ ---
SECE: Best Engineering College in Tamil Nadu
Skip to content
Sri Eshwar Engineering College
International Internship 2026 @ SECE
Campus tour
Admission 2025
TNEA
CODE
2739
TNEACODE
2739
Campus tour
Admissions
TNEACODE
2739
TNEACODE
2739
Campus tour
Admission 2025
TNEACODE
2739
TOPPERS' TOP CHOICE.
EXPERIENTIAL LEARNING.
FANTABULOUS CAMPUS.
Sri Eshwar
is the most preferred institution for high ranking students. With industry-relevant curriculum, project based learning, high energy faculty, corporate-like facilities, best amenities and vibrant activities, Sri Eshwar is the most sought after institution for high quality and holistic education. What’s more? We are proud to emerge, every year, as a
TOP PLACEMENTS COLLEGE!
Slot

In [8]:
import os
import shutil
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Initializing the sentence-transformers embedding engine...")
# 1. Load an open-source, high-efficiency embedding model from Hugging Face
# This model converts text strings into 384-dimensional mathematical vectors
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Set up a local directory path to store our persistent vector database records
db_dir = "./sece_vector_db"

# Clear out any existing old database folder to prevent configuration mismatches
if os.path.exists(db_dir):
    shutil.rmtree(db_dir)

print("Building ChromaDB vector database index from SECE text chunks...")
# 3. Feed the chunks into ChromaDB and calculate their geometric vector values
vector_db = Chroma.from_documents(
    documents=sece_chunks,
    embedding=embedding_function,
    persist_directory=db_dir
)

print("\nStep 3 Complete!")
print(f"Vector Database successfully created and saved locally at: '{db_dir}'")

# Quick Semantic Search Validation Check
print("\n--- RUNNING VECTOR DB TEST SEARCH ---")
query = "What companies visit for placements?"
matching_docs = vector_db.similarity_search(query, k=2)

print(f"Test Query: '{query}'")
print(f"Top Matched Chunk Found:\n{matching_docs[0].page_content}")
print("---------------------------------------")

/tmp/ipykernel_771/3173844580.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Initializing the sentence-transformers embedding engine...


/tmp/ipykernel_771/3173844580.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building ChromaDB vector database index from SECE text chunks...

Step 3 Complete!
Vector Database successfully created and saved locally at: './sece_vector_db'

--- RUNNING VECTOR DB TEST SEARCH ---
Test Query: 'What companies visit for placements?'
Top Matched Chunk Found:
Placement Training Facility
As a top placements college, we pride ourselves in providing exclusive placement training facilities. With more than four 200+ seating capacity facilities, four 60+ seating capacity training rooms, we can accommodate about 1000 students to undergo dedicated hands-on placement training at a time. With such exclusive training facilities paired with industry mentorship and high quality students, Sri Eshwar continues to top in placements every year in Tamil Nadu!.
Synapse Studio
Synapse Studio, an IT auditorium provides students with a hands-on learning experience. Industry experts and senior professionals utilize this space to conduct highly engaging hands-on sessions. dents to get into the

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Checking environment variables and loading Gemma weights...")

# 1. EMERGENCY CHECK: If the runtime dropped the model from memory, reload it securely
if 'tokenizer' not in globals() or 'model' not in globals():
    print("Gemma was not found in memory. Re-initializing model weights onto the T4 GPU...")
    model_id = "google/gemma-1.1-2b-it"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
else:
    print("Gemma is already securely sitting in active memory!")

print("\nAssembling the RAG Retriever-Generator engine...")

def ask_sece_bot(user_query: str):
    """
    RAG system that pulls facts from ChromaDB and passes them to Gemma to generate a safe answer.
    """
    # 2. RETRIEVER: Search ChromaDB for the top 3 closest matching chunks to our question
    matching_docs = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n".join([doc.page_content for doc in matching_docs])

    # 3. PROMPT ENGINEERING: Construct an explicit system prompt enclosing our retrieved facts
    rag_prompt = f"""You are an official AI academic assistant for Sri Eshwar College of Engineering (SECE).
Your task is to answer the user's question accurately using ONLY the verified context provided below.
If the answer cannot be found in the context, politely say that you do not have that specific information.

CONSTRAINTS: Do not guess or hallucinate. Keep the response professional and direct.

VERIFIED COLLEGE CONTEXT:
{retrieved_context}

USER QUESTION:
{user_query}

YOUR ACCURATE ANSWER:"""

    # 4. CHAT TEMPLATE & TOKENIZATION
    chat_structure = [{"role": "user", "content": rag_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(chat_structure, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    # 5. GENERATION: Let Gemma read the facts and respond safely
    generated_tokens = model.generate(
        **model_inputs,
        max_new_tokens=250,
        temperature=0.1,  # Ultra low temperature to make sure it sticks strictly to the facts
        do_sample=True
    )

    # 6. DECODE OUTPUT
    final_answer = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

    print("\n" + "="*60)
    print(f"🤖 SECE BOT ANSWER FOR: '{user_query}'")
    print("="*60)
    print(final_answer)
    print("="*60 + "\n")

# ========================================================
# TEST RUNNING YOUR COMPLETE PROJECT
# ========================================================
ask_sece_bot("Tell me about the Computer Science and Engineering department courses.")
ask_sece_bot("What are the core vision and mission values of Sri Eshwar College?")

Checking environment variables and loading Gemma weights...
Gemma was not found in memory. Re-initializing model weights onto the T4 GPU...


config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Assembling the RAG Retriever-Generator engine...

🤖 SECE BOT ANSWER FOR: 'Tell me about the Computer Science and Engineering department courses.'
user
You are an official AI academic assistant for Sri Eshwar College of Engineering (SECE). 
Your task is to answer the user's question accurately using ONLY the verified context provided below. 
If the answer cannot be found in the context, politely say that you do not have that specific information.

CONSTRAINTS: Do not guess or hallucinate. Keep the response professional and direct.

VERIFIED COLLEGE CONTEXT:
UG PROGRAMMES
ENGINEERING
B.E.Computer Science and Engineering
B.E.Computer Science and Engineering (Artificial Intelligence & Machine Learning)
B.E.Computer and Communication Engineering
B.E.Computer Science and Engineering (Cyber Security)
B.E.Electronics and Communication
Engineering
B.E.Electrical and Electronics Engineering
B.E.Mechanical Engineering
TECHNOLOGY
B.Tech.Artificial Intelligence and Data Science
B.Tech.Computer Sci